# 10 · Recomendaciones operativas para redistribución

Este notebook convierte la señal técnica del XGBoost congelado en **candidatas de redistribución explicables y sujetas a factibilidad**. Trabaja sobre las cuatro entradas exportadas por el notebook 09 y no utiliza etiquetas reales, errores individuales ni información futura.

Las salidas constituyen un prototipo retrospectivo de apoyo a decisiones. No representan una planificación definitiva de rutas ni sustituyen restricciones reales de flota, personal, tráfico o ventanas de servicio.

## Separación de responsabilidades

1. **Señal predictiva:** clase y probabilidades del modelo congelado.
2. **Factibilidad operativa:** capacidad, bicicletas, anclajes, distancia y balance entre origen y destino.
3. **Explicación:** factores SHAP globales de la clase y contexto actual de la estación.

`critical_risk_score` se usa exclusivamente para ordenar candidatas dentro de cada hora y acción. No filtra observaciones, no determina cantidades y no modifica umbrales. Para filas estables, `critical_risk_type` se ignora por completo.

## Entorno, rutas y parámetros de política

In [ ]:
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
MODEL_DATA_DIR = PROJECT_ROOT / 'notebooks' / 'Datos modelado'
INPUT_DIR = MODEL_DATA_DIR / 'interpretabilidad_modelo_final'
OUTPUT_DIR = MODEL_DATA_DIR / 'recomendaciones_operativas'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_COLUMN = 'fecha_hora_local'
CLASS_NAMES = {0: 'estable', 1: 'riesgo_vaciado', 2: 'riesgo_saturacion'}
CRITICAL_HOURS = [8, 18, 19]

# Parámetros de escenario operativo: son decisiones de negocio, no valores ajustados con el test.
OPERATIONAL_MIN_RATIO = 0.30
OPERATIONAL_MAX_RATIO = 0.70
MAX_PAIR_DISTANCE_KM = 3.0
MAX_UNITS_PER_TRANSFER = 10
TOP_STATIONS_FOR_MONITORING = 20

assert 0 < OPERATIONAL_MIN_RATIO < OPERATIONAL_MAX_RATIO < 1
assert MAX_PAIR_DISTANCE_KM > 0
assert MAX_UNITS_PER_TRANSFER >= 1

print('Entradas:', INPUT_DIR)
print('Salidas:', OUTPUT_DIR)
print({
    'ocupacion_minima_objetivo': OPERATIONAL_MIN_RATIO,
    'ocupacion_maxima_objetivo': OPERATIONAL_MAX_RATIO,
    'distancia_maxima_km': MAX_PAIR_DISTANCE_KM,
    'unidades_maximas_por_transferencia': MAX_UNITS_PER_TRANSFER,
})

## Carga y auditoría de las cuatro entradas

Las métricas históricas de horas y estaciones se incorporan solo como contexto de cautela. No entran en el ranking ni en el cálculo de unidades.

In [ ]:
input_paths = {
    'predictions': INPUT_DIR / 'entrada_10_predicciones_y_contexto.csv',
    'factors': INPUT_DIR / 'entrada_10_factores_por_clase.csv',
    'stations': INPUT_DIR / 'entrada_10_estaciones_sensibles.csv',
    'hours': INPUT_DIR / 'entrada_10_horas_sensibles.csv',
}
for input_name, input_path in input_paths.items():
    if not input_path.exists():
        raise FileNotFoundError(f'Falta la entrada {input_name}: {input_path}')

predictions = pd.read_csv(
    input_paths['predictions'], encoding='utf-8-sig', low_memory=False,
    dtype={'station_number': 'string'},
)
factors = pd.read_csv(input_paths['factors'], encoding='utf-8-sig')
station_sensitivity = pd.read_csv(input_paths['stations'], encoding='utf-8-sig')
hour_sensitivity = pd.read_csv(input_paths['hours'], encoding='utf-8-sig')
predictions[TIME_COLUMN] = pd.to_datetime(predictions[TIME_COLUMN], errors='raise')
predictions['prediction'] = predictions['prediction'].astype('int8')
predictions['station_id'] = predictions['station_id'].astype(int)

required_prediction_columns = {
    TIME_COLUMN, 'station_id', 'capacity', 'bikes_available', 'docks_available',
    'reservations_count', 'occupancy_ratio', 'prediction', 'confidence',
    'probability_0', 'probability_1', 'probability_2',
    'predicted_risk_name', 'model_action_signal', 'critical_risk_score',
    'critical_risk_type', 'latitude', 'longitude',
}
assert required_prediction_columns.issubset(predictions.columns)
assert 'risk_class_1h' not in predictions.columns
assert 'is_error' not in predictions.columns
assert not predictions.duplicated([TIME_COLUMN, 'station_id']).any()
assert predictions['prediction'].isin(CLASS_NAMES).all()
assert predictions.loc[predictions['prediction'].eq(0), 'model_action_signal'].eq('monitorizar').all()
assert len(predictions) == 379_104

print('Predicciones y contexto:', f'{len(predictions):,}')
print('Factores SHAP:', len(factors))
print('Estaciones sensibles:', len(station_sensitivity))
print('Horas sensibles:', len(hour_sensitivity))

## Brecha operativa y candidatas técnicas

El rango objetivo 30–70 % crea un margen de seguridad. Una señal de vaciado solo solicita bicicletas si la estación está por debajo del mínimo; una señal de saturación solo ofrece bicicletas si supera el máximo. Cuando existe riesgo predictivo pero no brecha actual, la acción pasa a monitorización programada.

In [ ]:
work = predictions.copy()
work['hour'] = work[TIME_COLUMN].dt.hour
work['month'] = work[TIME_COLUMN].dt.to_period('M').astype(str)
work['target_min_bikes'] = np.ceil(work['capacity'] * OPERATIONAL_MIN_RATIO).astype(int)
work['target_max_bikes'] = np.floor(work['capacity'] * OPERATIONAL_MAX_RATIO).astype(int)
work['free_docks_conservative'] = np.maximum(
    0, work['docks_available'].fillna(0) - work['reservations_count'].fillna(0)
).astype(int)
work['receiver_need_units'] = np.minimum(
    np.maximum(0, work['target_min_bikes'] - work['bikes_available']),
    work['free_docks_conservative'],
).astype(int)
work['donor_supply_units'] = np.maximum(
    0, work['bikes_available'] - work['target_max_bikes']
).astype(int)

# Las estaciones estables pueden actuar como apoyo seguro, pero nunca originan una candidata técnica.
work['can_receive'] = (
    work['receiver_need_units'].gt(0) & work['prediction'].isin([0, 1])
)
work['can_donate'] = (
    work['donor_supply_units'].gt(0) & work['prediction'].isin([0, 2])
)

technical_candidates = work.loc[work['prediction'].isin([1, 2])].copy()
technical_candidates['requested_units'] = np.where(
    technical_candidates['prediction'].eq(1),
    technical_candidates['receiver_need_units'],
    technical_candidates['donor_supply_units'],
).astype(int)
technical_candidates['candidate_role'] = np.where(
    technical_candidates['prediction'].eq(1), 'receiver', 'donor'
)
technical_candidates['priority_rank_within_hour_and_action'] = (
    technical_candidates.groupby([TIME_COLUMN, 'prediction'])['critical_risk_score']
    .rank(method='first', ascending=False).astype(int)
)
technical_candidates['score_use'] = 'solo_ordenacion_sin_umbral'

assert len(technical_candidates) == int(work['prediction'].ne(0).sum())
assert not technical_candidates['prediction'].eq(0).any()
assert technical_candidates['score_use'].eq('solo_ordenacion_sin_umbral').all()
print('Candidatas técnicas:', f'{len(technical_candidates):,}')
display(technical_candidates.groupby(['prediction', 'candidate_role']).agg(
    rows=('station_id', 'size'), requested_units=('requested_units', 'sum'),
    rows_with_immediate_gap=('requested_units', lambda values: int(values.gt(0).sum())),
).reset_index())

## Explicaciones SHAP y avisos de cautela

La explicación es global para la clase predicha, no una explicación SHAP individual. Las horas y estaciones sensibles generan notas de monitorización y no alteran `critical_risk_score`, prioridad o unidades.

In [ ]:
direction_labels = {
    'valores_altos_aumentan_contribucion': 'valores altos suelen aumentar la señal',
    'valores_altos_reducen_contribucion': 'valores altos suelen reducir la señal',
    'sin_direccion_monotona_clara': 'efecto no monótono',
}
class_explanations = {}
for class_value in [1, 2]:
    top = (
        factors.loc[factors['class_value'].eq(class_value)]
        .sort_values('shap_rank_within_class').head(5)
    )
    pieces = []
    for row in top.itertuples(index=False):
        direction = direction_labels.get(row.direction_association, 'interacción no lineal o categórica')
        pieces.append(f'{row.original_feature} ({direction})')
    class_explanations[class_value] = '; '.join(pieces)

top_monitoring_stations = set(
    station_sensitivity.nlargest(TOP_STATIONS_FOR_MONITORING, 'critical_false_negatives')['station_id']
)
station_context_columns = [
    'station_id', 'critical_support', 'critical_false_negatives',
    'critical_false_negative_rate', 'enough_support_empty', 'enough_support_full',
]
station_context = station_sensitivity[station_context_columns].copy()
hour_context = hour_sensitivity[[
    'hour', 'critical_support', 'critical_false_negatives', 'critical_false_negative_rate'
]].rename(columns={
    'critical_support': 'hour_critical_support',
    'critical_false_negatives': 'hour_critical_false_negatives',
    'critical_false_negative_rate': 'hour_critical_false_negative_rate',
})
technical_candidates = technical_candidates.merge(
    station_context, on='station_id', how='left', validate='many_to_one',
).merge(hour_context, on='hour', how='left', validate='many_to_one')
technical_candidates['sensitive_hour_flag'] = technical_candidates['hour'].isin(CRITICAL_HOURS)
technical_candidates['sensitive_station_flag'] = technical_candidates['station_id'].isin(top_monitoring_stations)
technical_candidates['sensitivity_use'] = 'solo_aviso_no_modifica_recomendacion'
technical_candidates['class_shap_explanation'] = technical_candidates['prediction'].map(class_explanations)

def candidate_explanation(row) -> str:
    current = (
        f'estado actual {int(row.bikes_available)}/{int(row.capacity)} bicicletas '
        f'(ocupación {row.occupancy_ratio:.1%}); brecha operativa {int(row.requested_units)}'
    )
    cautions = []
    if row.sensitive_hour_flag:
        cautions.append('hora históricamente sensible')
    if row.sensitive_station_flag:
        cautions.append('estación con errores históricos relevantes')
    caution_text = '; '.join(cautions) if cautions else 'sin aviso histórico especial'
    return (
        f'Señal {CLASS_NAMES[int(row.prediction)]}; {current}. '
        f'Factores globales de clase: {row.class_shap_explanation}. '
        f'Cautela: {caution_text}. Asociación predictiva, no causal.'
    )

technical_candidates['operational_explanation'] = technical_candidates.apply(
    candidate_explanation, axis=1
)
display(pd.DataFrame([
    {'class_value': class_value, 'class_name': CLASS_NAMES[class_value], 'explanation': explanation}
    for class_value, explanation in class_explanations.items()
]))

## Emparejamiento geográfico con balance de bicicletas

Para cada hora se construyen orígenes con excedente y destinos con déficit. Se priorizan parejas donde ambos extremos tienen riesgos complementarios; después pueden utilizarse estaciones estables con margen seguro. El algoritmo es voraz y explicable, no una optimización global de rutas.

In [ ]:
TRANSFER_COLUMNS = [
    TIME_COLUMN, 'donor_station_id', 'donor_station_name', 'donor_prediction',
    'donor_role', 'receiver_station_id', 'receiver_station_name',
    'receiver_prediction', 'receiver_role', 'units', 'distance_km',
    'pair_type', 'risk_score_for_ordering',
]

def haversine_matrix(donors: pd.DataFrame, receivers: pd.DataFrame) -> np.ndarray:
    earth_radius_km = 6371.0088
    donor_lat = np.radians(donors['latitude'].to_numpy(dtype=float))[:, None]
    donor_lon = np.radians(donors['longitude'].to_numpy(dtype=float))[:, None]
    receiver_lat = np.radians(receivers['latitude'].to_numpy(dtype=float))[None, :]
    receiver_lon = np.radians(receivers['longitude'].to_numpy(dtype=float))[None, :]
    delta_lat = receiver_lat - donor_lat
    delta_lon = receiver_lon - donor_lon
    a = (
        np.sin(delta_lat / 2) ** 2
        + np.cos(donor_lat) * np.cos(receiver_lat) * np.sin(delta_lon / 2) ** 2
    )
    return earth_radius_km * 2 * np.arctan2(np.sqrt(a), np.sqrt(np.maximum(0, 1 - a)))

def match_one_timestamp(timestamp: pd.Timestamp, frame: pd.DataFrame) -> list[dict]:
    donors = frame.loc[frame['can_donate']].copy().reset_index(drop=True)
    receivers = frame.loc[frame['can_receive']].copy().reset_index(drop=True)
    if donors.empty or receivers.empty:
        return []

    distances = haversine_matrix(donors, receivers)
    donor_remaining = donors['donor_supply_units'].to_numpy(dtype=int).copy()
    receiver_remaining = receivers['receiver_need_units'].to_numpy(dtype=int).copy()
    pair_candidates = []

    for donor_position, receiver_position in np.argwhere(distances <= MAX_PAIR_DISTANCE_KM):
        donor = donors.iloc[donor_position]
        receiver = receivers.iloc[receiver_position]
        if int(donor.station_id) == int(receiver.station_id):
            continue
        donor_is_risk = int(donor.prediction) == 2
        receiver_is_risk = int(receiver.prediction) == 1
        if not donor_is_risk and not receiver_is_risk:
            continue
        both_risk = donor_is_risk and receiver_is_risk
        risk_score = max(
            float(donor.critical_risk_score) if donor_is_risk else -1.0,
            float(receiver.critical_risk_score) if receiver_is_risk else -1.0,
        )
        pair_candidates.append((
            0 if both_risk else 1, -risk_score, float(distances[donor_position, receiver_position]),
            int(donor_position), int(receiver_position), risk_score,
        ))

    transfers = []
    for _, _, distance_km, donor_position, receiver_position, risk_score in sorted(pair_candidates):
        units = min(
            donor_remaining[donor_position], receiver_remaining[receiver_position],
            MAX_UNITS_PER_TRANSFER,
        )
        if units <= 0:
            continue
        donor = donors.iloc[donor_position]
        receiver = receivers.iloc[receiver_position]
        donor_remaining[donor_position] -= units
        receiver_remaining[receiver_position] -= units
        donor_is_risk = int(donor.prediction) == 2
        receiver_is_risk = int(receiver.prediction) == 1
        pair_type = (
            'riesgos_complementarios' if donor_is_risk and receiver_is_risk
            else 'apoyo_estable_a_vaciado' if receiver_is_risk
            else 'saturacion_a_estacion_estable'
        )
        transfers.append({
            TIME_COLUMN: timestamp,
            'donor_station_id': int(donor.station_id),
            'donor_station_name': donor.station_name,
            'donor_prediction': int(donor.prediction),
            'donor_role': 'riesgo_saturacion' if donor_is_risk else 'apoyo_estable_con_excedente',
            'receiver_station_id': int(receiver.station_id),
            'receiver_station_name': receiver.station_name,
            'receiver_prediction': int(receiver.prediction),
            'receiver_role': 'riesgo_vaciado' if receiver_is_risk else 'apoyo_estable_con_deficit',
            'units': int(units),
            'distance_km': distance_km,
            'pair_type': pair_type,
            'risk_score_for_ordering': risk_score,
        })
    return transfers


In [ ]:
transfer_parts = []
for timestamp, timestamp_frame in work.groupby(TIME_COLUMN, sort=True):
    transfer_parts.extend(match_one_timestamp(timestamp, timestamp_frame))
transfers = pd.DataFrame(transfer_parts, columns=TRANSFER_COLUMNS)
if not transfers.empty:
    transfers[TIME_COLUMN] = pd.to_datetime(transfers[TIME_COLUMN])

receiver_allocations = (
    transfers.groupby([TIME_COLUMN, 'receiver_station_id'], as_index=False)['units'].sum()
    .rename(columns={'receiver_station_id': 'station_id', 'units': 'receiver_units_assigned'})
) if not transfers.empty else pd.DataFrame(columns=[TIME_COLUMN, 'station_id', 'receiver_units_assigned'])
donor_allocations = (
    transfers.groupby([TIME_COLUMN, 'donor_station_id'], as_index=False)['units'].sum()
    .rename(columns={'donor_station_id': 'station_id', 'units': 'donor_units_assigned'})
) if not transfers.empty else pd.DataFrame(columns=[TIME_COLUMN, 'station_id', 'donor_units_assigned'])

technical_candidates = technical_candidates.merge(
    receiver_allocations, on=[TIME_COLUMN, 'station_id'], how='left', validate='one_to_one'
).merge(donor_allocations, on=[TIME_COLUMN, 'station_id'], how='left', validate='one_to_one')
technical_candidates[['receiver_units_assigned', 'donor_units_assigned']] = technical_candidates[[
    'receiver_units_assigned', 'donor_units_assigned'
]].fillna(0).astype(int)
technical_candidates['assigned_units'] = np.where(
    technical_candidates['prediction'].eq(1),
    technical_candidates['receiver_units_assigned'],
    technical_candidates['donor_units_assigned'],
).astype(int)
technical_candidates['uncovered_units'] = np.maximum(
    0, technical_candidates['requested_units'] - technical_candidates['assigned_units']
).astype(int)
technical_candidates['recommendation_status'] = np.select(
    [
        technical_candidates['requested_units'].eq(0),
        technical_candidates['assigned_units'].eq(0),
        technical_candidates['assigned_units'].lt(technical_candidates['requested_units']),
    ],
    [
        'monitorizar_sin_brecha_operativa_actual',
        'sin_pareja_factible_en_radio',
        'cobertura_parcial',
    ],
    default='transferencia_asignada',
)
unresolved_candidates = technical_candidates.loc[
    ~technical_candidates['recommendation_status'].eq('transferencia_asignada')
].copy()

print('Transferencias factibles:', f'{len(transfers):,}')
print('Bicicletas redistribuidas:', f'{int(transfers.units.sum()) if not transfers.empty else 0:,}')
display(technical_candidates['recommendation_status'].value_counts().rename('rows').to_frame())

## Validación de restricciones

Las aserciones comprueban distancia, capacidad, balance y ausencia de parejas estable–estable. El score de riesgo no participa en ninguna fórmula de cantidad.

In [ ]:
if not transfers.empty:
    assert transfers['units'].between(1, MAX_UNITS_PER_TRANSFER).all()
    assert transfers['distance_km'].le(MAX_PAIR_DISTANCE_KM + 1e-9).all()
    assert transfers['donor_station_id'].ne(transfers['receiver_station_id']).all()
    assert ~transfers['pair_type'].eq('estable_a_estable').any()

    donor_check = (
        transfers.groupby([TIME_COLUMN, 'donor_station_id'], as_index=False)['units'].sum()
        .merge(
            work[[TIME_COLUMN, 'station_id', 'donor_supply_units']],
            left_on=[TIME_COLUMN, 'donor_station_id'],
            right_on=[TIME_COLUMN, 'station_id'], how='left', validate='one_to_one',
        )
    )
    receiver_check = (
        transfers.groupby([TIME_COLUMN, 'receiver_station_id'], as_index=False)['units'].sum()
        .merge(
            work[[TIME_COLUMN, 'station_id', 'receiver_need_units']],
            left_on=[TIME_COLUMN, 'receiver_station_id'],
            right_on=[TIME_COLUMN, 'station_id'], how='left', validate='one_to_one',
        )
    )
    assert donor_check['units'].le(donor_check['donor_supply_units']).all()
    assert receiver_check['units'].le(receiver_check['receiver_need_units']).all()
    assert int(donor_check['units'].sum()) == int(receiver_check['units'].sum()) == int(transfers['units'].sum())

assert technical_candidates['assigned_units'].le(technical_candidates['requested_units']).all()
assert technical_candidates.loc[technical_candidates['requested_units'].eq(0), 'assigned_units'].eq(0).all()
assert technical_candidates['priority_rank_within_hour_and_action'].ge(1).all()
assert technical_candidates['sensitivity_use'].eq('solo_aviso_no_modifica_recomendacion').all()
print('Restricciones de distancia, capacidad y balance verificadas.')

## Resúmenes operativos

Los resúmenes separan solicitudes, asignaciones y brechas no cubiertas. La sensibilidad histórica se conserva como columna de contexto, nunca como multiplicador de prioridad.

In [ ]:
def recommendation_summary(frame: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    return (
        frame.groupby(group_columns, dropna=False, as_index=False)
        .agg(
            candidate_rows=('station_id', 'size'),
            requested_units=('requested_units', 'sum'),
            assigned_units=('assigned_units', 'sum'),
            uncovered_units=('uncovered_units', 'sum'),
            mean_risk_score=('critical_risk_score', 'mean'),
            sensitive_hour_rows=('sensitive_hour_flag', 'sum'),
            sensitive_station_rows=('sensitive_station_flag', 'sum'),
        )
    )

summary_by_timestamp = recommendation_summary(
    technical_candidates, [TIME_COLUMN, 'candidate_role']
)
summary_by_hour = recommendation_summary(
    technical_candidates, ['hour', 'candidate_role']
)
summary_by_station = recommendation_summary(
    technical_candidates, ['station_id', 'station_number', 'station_name', 'candidate_role']
)

transfer_summary_by_timestamp = (
    transfers.groupby(TIME_COLUMN, as_index=False)
    .agg(
        transfer_pairs=('units', 'size'),
        transferred_units=('units', 'sum'),
        mean_distance_km=('distance_km', 'mean'),
        max_distance_km=('distance_km', 'max'),
        stable_support_pairs=('pair_type', lambda values: int(values.ne('riesgos_complementarios').sum())),
    )
) if not transfers.empty else pd.DataFrame(columns=[
    TIME_COLUMN, 'transfer_pairs', 'transferred_units', 'mean_distance_km',
    'max_distance_km', 'stable_support_pairs',
])

policy_table = pd.DataFrame([
    {'parameter': 'operational_min_ratio', 'value': OPERATIONAL_MIN_RATIO, 'source': 'politica_operativa_no_ajustada'},
    {'parameter': 'operational_max_ratio', 'value': OPERATIONAL_MAX_RATIO, 'source': 'politica_operativa_no_ajustada'},
    {'parameter': 'max_pair_distance_km', 'value': MAX_PAIR_DISTANCE_KM, 'source': 'restriccion_logistica_de_escenario'},
    {'parameter': 'max_units_per_transfer', 'value': MAX_UNITS_PER_TRANSFER, 'source': 'restriccion_logistica_de_escenario'},
    {'parameter': 'risk_score_threshold', 'value': 'none', 'source': 'score_solo_para_ordenar'},
])
display(policy_table)
display(summary_by_hour.loc[summary_by_hour['hour'].isin(CRITICAL_HOURS)])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
hour_plot = summary_by_hour.groupby('hour', as_index=False)[['requested_units', 'assigned_units']].sum()
axes[0].plot(hour_plot['hour'], hour_plot['requested_units'], marker='o', label='Solicitadas')
axes[0].plot(hour_plot['hour'], hour_plot['assigned_units'], marker='o', label='Asignadas')
for hour in CRITICAL_HOURS:
    axes[0].axvline(hour, color='#D97904', alpha=0.35, linestyle='--')
axes[0].set_title('Unidades por hora del día')
axes[0].set_xlabel('Hora')
axes[0].set_ylabel('Bicicletas')
axes[0].legend()
axes[0].grid(alpha=0.25)

top_add = (
    summary_by_station.loc[summary_by_station['candidate_role'].eq('receiver')]
    .nlargest(12, 'assigned_units').sort_values('assigned_units')
)
axes[1].barh(top_add['station_name'], top_add['assigned_units'], color='#0B6E99')
axes[1].set_title('Estaciones con más reposiciones asignadas')
axes[1].set_xlabel('Bicicletas')

top_remove = (
    summary_by_station.loc[summary_by_station['candidate_role'].eq('donor')]
    .nlargest(12, 'assigned_units').sort_values('assigned_units')
)
axes[2].barh(top_remove['station_name'], top_remove['assigned_units'], color='#A23B72')
axes[2].set_title('Estaciones con más retiradas asignadas')
axes[2].set_xlabel('Bicicletas')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'resumen_recomendaciones_operativas.png', dpi=160, bbox_inches='tight')
plt.show()

## Exportación y auditoría

Las recomendaciones conservan la señal, la factibilidad y la explicación en columnas separadas para que puedan auditarse de forma independiente.

In [ ]:
class_explanation_table = pd.DataFrame([
    {'class_value': class_value, 'class_name': CLASS_NAMES[class_value], 'global_shap_explanation': explanation}
    for class_value, explanation in class_explanations.items()
])
exports = {
    'candidatos_tecnicos_priorizados.csv': technical_candidates,
    'transferencias_factibles.csv': transfers,
    'candidatos_no_resueltos.csv': unresolved_candidates,
    'resumen_por_timestamp.csv': summary_by_timestamp,
    'resumen_transferencias_por_timestamp.csv': transfer_summary_by_timestamp,
    'resumen_por_hora.csv': summary_by_hour,
    'resumen_por_estacion.csv': summary_by_station,
    'politica_operativa.csv': policy_table,
    'explicaciones_shap_por_clase.csv': class_explanation_table,
}
for file_name, table in exports.items():
    table.to_csv(OUTPUT_DIR / file_name, index=False, encoding='utf-8-sig')

total_units = int(transfers['units'].sum()) if not transfers.empty else 0
audit_table = pd.DataFrame([
    {'control': 'input_rows', 'value': len(predictions)},
    {'control': 'target_column_present', 'value': False},
    {'control': 'is_error_column_present', 'value': False},
    {'control': 'stable_rows_as_technical_candidates', 'value': 0},
    {'control': 'critical_risk_type_used_for_stable_action', 'value': False},
    {'control': 'risk_score_threshold', 'value': 'none'},
    {'control': 'risk_score_use', 'value': 'ordering_only'},
    {'control': 'test_metrics_used_in_priority', 'value': False},
    {'control': 'test_metrics_used_in_amount', 'value': False},
    {'control': 'sensitivity_metrics_use', 'value': 'monitoring_note_only'},
    {'control': 'operational_min_ratio', 'value': OPERATIONAL_MIN_RATIO},
    {'control': 'operational_max_ratio', 'value': OPERATIONAL_MAX_RATIO},
    {'control': 'max_pair_distance_km', 'value': MAX_PAIR_DISTANCE_KM},
    {'control': 'max_units_per_transfer', 'value': MAX_UNITS_PER_TRANSFER},
    {'control': 'technical_candidates', 'value': len(technical_candidates)},
    {'control': 'transfer_pairs', 'value': len(transfers)},
    {'control': 'transferred_units', 'value': total_units},
    {'control': 'model_retrained', 'value': False},
    {'control': 'thresholds_recalibrated', 'value': False},
    {'control': 'causal_interpretation', 'value': False},
])
audit_table.to_csv(OUTPUT_DIR / 'auditoria_recomendaciones_operativas.csv', index=False, encoding='utf-8-sig')

manifest = {
    'notebook': '10_recomendaciones_operativas.ipynb',
    'input_files': {name: str(path) for name, path in input_paths.items()},
    'policy': {
        'operational_min_ratio': OPERATIONAL_MIN_RATIO,
        'operational_max_ratio': OPERATIONAL_MAX_RATIO,
        'max_pair_distance_km': MAX_PAIR_DISTANCE_KM,
        'max_units_per_transfer': MAX_UNITS_PER_TRANSFER,
    },
    'risk_score_use': 'ordering_only_no_threshold',
    'matching_method': 'greedy_complementary_risks_then_safe_stable_support',
    'outputs': sorted(list(exports) + [
        'auditoria_recomendaciones_operativas.csv',
        'resumen_recomendaciones_operativas.png',
    ]),
}
with (OUTPUT_DIR / 'manifiesto_recomendaciones_operativas.json').open('w', encoding='utf-8') as file:
    json.dump(manifest, file, indent=2, ensure_ascii=False)

assert not bool(audit_table.loc[audit_table['control'].eq('target_column_present'), 'value'].iloc[0])
assert not bool(audit_table.loc[audit_table['control'].eq('thresholds_recalibrated'), 'value'].iloc[0])
assert int(audit_table.loc[audit_table['control'].eq('stable_rows_as_technical_candidates'), 'value'].iloc[0]) == 0
display(audit_table)
print('Recomendaciones operativas guardadas en:', OUTPUT_DIR)

## Lectura y limitaciones

- Una transferencia asignada satisface restricciones del escenario, pero aún debe convertirse en ruta y turno de trabajo.
- Una candidata sin brecha actual debe monitorizarse: el modelo anticipa riesgo a una hora, pero la disponibilidad presente todavía no justifica movimiento inmediato.
- Una candidata sin pareja cercana no debe descartarse; puede requerir ampliar radio, usar almacén central o coordinar varias paradas. Esas alternativas no se ajustan con el test.
- Las estaciones estables solo actúan como apoyo cuando conservan el margen 30–70 %. Nunca se genera una acción porque `critical_risk_type` indique un riesgo relativo en una fila estable.
- Las explicaciones SHAP describen el modelo y no prueban causalidad.
- Para una aplicación real, los parámetros de política deben validarse con responsables de operación y datos actuales, y el emparejamiento voraz debería evolucionar a una optimización de rutas con vehículos, tiempos y costes.